B19. Ранний останов: ретроспективная оптимизация правила (Liu 2026, Michael 2023).....


In [1]:
import sys, os, re

def _build_data_tools():
    import json
    import os
    import re
    import warnings
    import numpy as np
    import pandas as pd
    from scipy import stats
    warnings.filterwarnings('ignore')

    def path_join(*parts):
        return os.path.normpath(os.path.join(*parts))

    def normalize_newlines(frame):
        for column in frame.select_dtypes(include=['object', 'string']).columns:
            frame[column] = frame[column].map(lambda value: value.replace('\r\n', '\n').replace('\r', '\n') if isinstance(value, str) else value)
        return frame
    from pathlib import Path
    _start = Path.cwd().resolve()
    _repo = next((p for p in (_start, *_start.parents) if (p / 'experiments/clinical_protocols').is_dir() and (p / 'data').is_dir()), None)
    if _repo is None:
        raise FileNotFoundError('Run the notebook from inside the repository.')
    if os.name == 'nt' and (not str(_repo).startswith(chr(92) * 2 + '?' + chr(92))):
        _repo = Path(chr(92) * 2 + '?' + chr(92) + str(_repo))
    ROOT = str(_repo)
    HERE = str(_repo / 'analyses/protocol_design')
    OUT = path_join(HERE, 'results')
    os.makedirs(OUT, exist_ok=True)
    PBASE_DIR = 'experiments/clinical_protocols/delphi_panels/heterogeneous_gpt120_organizer_complete'
    HOMOG_DS_DIR = 'experiments/homogeneous_panels/three_deepseekv4_members_complete'
    NB = 'experiments/clinical_protocols/single_model_baselines/baseline_%s/llm_as_judge/baseline_%s_%s_judge.csv'
    USC = 'experiments/token_budget_controls/universal_self_consistency_gpt_seven_samples/llm_as_judge/usc_gpt_sc%d_judge.csv'
    HOMOG_DIRS = {'gpt': 'experiments/homogeneous_panels/three_gpt_members', 'qwen': 'experiments/homogeneous_panels/three_qwen_members', 'gemini': 'experiments/homogeneous_panels/three_gemini_members'}
    PANELS_OTHER_DIR = {'P-weak': 'experiments/clinical_protocols/delphi_panels/panel_qwen_organizer_gemini_deepseek_pro', 'P-mixed': 'experiments/clinical_protocols/delphi_panels/panel_deepseek_organizer_gemini_glm', 'P-strong': 'experiments/clinical_protocols/delphi_panels/panel_deepseek_organizer_glm_gpt120'}
    PANEL_FILE = {'P-weak': 'delphi_pweak_noroles_judge.csv', 'P-mixed': 'delphi_pmixed_noroles_judge.csv', 'P-strong': 'delphi_pstrong_noroles_judge.csv'}
    SINGLES = ['gpt', 'qwen', 'deepseekv4', 'deepseekv4pro', 'gemini', 'glm', 'kimi']
    COMPOSITION = {'P-base': ['gpt', 'qwen', 'deepseekv4'], 'P-weak': ['qwen', 'gemini', 'deepseekv4pro'], 'P-mixed': ['deepseekv4', 'gemini', 'glm'], 'P-strong': ['deepseekv4', 'glm', 'gpt'], 'homog gpt': ['gpt'] * 3, 'homog qwen': ['qwen'] * 3, 'homog deepseekv4': ['deepseekv4'] * 3, 'homog gemini': ['gemini'] * 3}
    HETERO = {'P-base': 1, 'P-weak': 1, 'P-mixed': 1, 'P-strong': 1, 'homog gpt': 0, 'homog qwen': 0, 'homog deepseekv4': 0, 'homog gemini': 0}
    PANELS = list(COMPOSITION)
    LADDER = ['baseline gpt', 'USC@3 gpt', 'USC@5 gpt', 'USC@7 gpt', 'homog gpt', 'homog deepseekv4', 'P-base']

    def arm_paths(design='noroles'):
        p = {}
        for m in SINGLES:
            p['baseline ' + m] = NB % (m, design, m)
        if design == 'noroles':
            for k in (3, 5, 7):
                p['USC@%d gpt' % k] = USC % k
            for tag, d in HOMOG_DIRS.items():
                p['homog ' + tag] = d + '/llm_as_judge/delphi_homog_%s_noroles_judge.csv' % tag
            p['homog deepseekv4'] = HOMOG_DS_DIR + '/llm_as_judge/delphi_homog_deepseekv4_noroles_judge.csv'
            for k, d in PANELS_OTHER_DIR.items():
                p[k] = d + '/llm_as_judge/' + PANEL_FILE[k]
        p['P-base'] = PBASE_DIR + '/llm_as_judge/delphi_base_%s_judge.csv' % design
        return p
    ROUNDS_DIR = {'P-base': (PBASE_DIR, 'delphi_base_noroles_protocol_r%d_judge.csv'), 'P-weak': (PANELS_OTHER_DIR['P-weak'], 'delphi_pweak_noroles_protocol_r%d_judge.csv'), 'P-mixed': (PANELS_OTHER_DIR['P-mixed'], 'delphi_pmixed_noroles_protocol_r%d_judge.csv'), 'P-strong': (PANELS_OTHER_DIR['P-strong'], 'delphi_pstrong_noroles_protocol_r%d_judge.csv'), 'homog gpt': (HOMOG_DIRS['gpt'], 'delphi_homog_gpt_noroles_protocol_r%d_judge.csv'), 'homog gemini': (HOMOG_DIRS['gemini'], 'delphi_homog_gemini_noroles_protocol_r%d_judge.csv'), 'homog deepseekv4': (HOMOG_DS_DIR, 'delphi_homog_deepseekv4_noroles_protocol_r%d_judge.csv')}
    RETEST = {'P-base': PBASE_DIR + '/llm_as_judge/judge_robustness/delphi_base_%s_retest_r%d_judge.csv', 'baseline gpt': 'experiments/clinical_protocols/single_model_baselines/baseline_gpt/llm_as_judge/judge_robustness/baseline_%s_gpt_retest_r%d_judge.csv', 'baseline qwen': 'experiments/clinical_protocols/single_model_baselines/baseline_qwen/llm_as_judge/judge_robustness/baseline_%s_qwen_retest_r%d_judge.csv', 'baseline deepseekv4': 'experiments/clinical_protocols/single_model_baselines/baseline_deepseekv4/llm_as_judge/judge_robustness/baseline_%s_deepseekv4_retest_r%d_judge.csv'}
    SHORT = {'P-base': PBASE_DIR + '/llm_as_judge/judge_robustness/delphi_base_%s_short_judge.csv', 'baseline gpt': 'experiments/clinical_protocols/single_model_baselines/baseline_gpt/llm_as_judge/judge_robustness/baseline_%s_gpt_short_judge.csv', 'baseline qwen': 'experiments/clinical_protocols/single_model_baselines/baseline_qwen/llm_as_judge/judge_robustness/baseline_%s_qwen_short_judge.csv', 'baseline deepseekv4': 'experiments/clinical_protocols/single_model_baselines/baseline_deepseekv4/llm_as_judge/judge_robustness/baseline_%s_deepseekv4_short_judge.csv'}
    JUDGE41 = {'baseline gpt': 'experiments/clinical_protocols/single_model_baselines/baseline_gpt/llm_as_judge/judge_gpt41/baseline_%s_gpt_judge.csv', 'baseline qwen': 'experiments/clinical_protocols/single_model_baselines/baseline_qwen/llm_as_judge/judge_gpt41/baseline_%s_qwen_judge.csv', 'baseline deepseekv4': 'experiments/clinical_protocols/single_model_baselines/baseline_deepseekv4/llm_as_judge/judge_gpt41/baseline_%s_deepseekv4_judge.csv'}
    CASE_COVARS = ['Type', 'Sub Clinical Specialties', 'Age Group', 'Genealogy', 'Diagnosis10', 'Vignette_race']
    DEXP = 'experiments/panel_parameter_sweeps'
    SWEEPS = {'experts': (DEXP + '/experts/llm_as_judge', 'delphi_experts_%s_judge.csv'), 'questionnaire_items': (DEXP + '/questionnaire_items/llm_as_judge', 'delphi_questionnaire_items_%s_judge.csv'), 'questions_per_expert': (DEXP + '/questions_per_expert/llm_as_judge', 'delphi_questions_per_expert_%s_judge.csv'), 'temperature': (DEXP + '/temperature/llm_as_judge', 'delphi_temperature_%s_judge.csv'), 'rounds': (DEXP + '/rounds/llm_as_judge', 'delphi_%s_judge.csv')}
    EXPERT_GROUPS = {'group1_10models': 10, 'group2_4models': 4, 'group3_5models': 5, 'group4_5models': 5, 'group5_7models': 7, 'group6_5models': 5, 'group7_6models': 6}
    EXT = 'experiments'
    MEDMCQA = {'baseline deepseekv4': EXT + '/external_benchmarks/medmcqa/baseline_deepseekv4/baseline_medmcqa_deepseekv4.csv', 'baseline gpt': EXT + '/external_benchmarks/medmcqa/baseline_gpt/baseline_medmcqa_gpt.csv', 'baseline qwen': EXT + '/external_benchmarks/medmcqa/baseline_qwen/baseline_medmcqa_qwen.csv', 'baseline alice': EXT + '/external_benchmarks/initial_runs/baseline_alice/baseline_medmcqa_alice.csv', 'baseline qwen36': EXT + '/external_benchmarks/initial_runs/baseline_qwen36/baseline_medmcqa_qwen36.csv', 'P-ac (gpt/qwen/ds)': EXT + '/external_benchmarks/medmcqa/panel_gpt120_organizer_qwen_deepseek/delphi_medmcqa_noroles.csv', 'P-ac (gpt/qwen/alice)': EXT + '/external_benchmarks/initial_runs/panel_gpt120_organizer_qwen_alice/delphi_medmcqa_noroles.csv', 'P-deepseektrio': EXT + '/external_benchmarks/initial_runs/three_deepseek_members/delphi_medmcqa_noroles.csv', 'P-base-like': EXT + '/external_benchmarks/initial_runs/heterogeneous_gpt120_organizer/delphi_medmcqa_noroles.csv'}

    def rd_any(rel, index=None):
        p = path_join(ROOT, rel)
        if not os.path.exists(p):
            return None
        with open(p, encoding='utf-8-sig', errors='replace') as fh:
            head = fh.readline()
        sep = ';' if head.count(';') > head.count(',') else ','
        d = normalize_newlines(pd.read_csv(p, sep=sep, encoding='utf-8-sig', low_memory=False))
        if 'table_source' in d.columns:
            d = d[d['table_source'].fillna('result_df') == 'result_df']
        if index and index in d.columns:
            d = d.dropna(subset=[index]).drop_duplicates(index).set_index(index)
        return d

    def sweep_arms(kind, verbose=True):
        import glob as _g
        folder, _ = SWEEPS[kind]
        out = {}
        for p in sorted(_g.glob(path_join(ROOT, folder, '*_judge.csv'))):
            name = os.path.basename(p)[:-len('_judge.csv')]
            d = rd(os.path.relpath(p, ROOT).replace('\\', '/'))
            if d is not None:
                out[name] = d
        if verbose:
            print('свип %s: плеч %d, кейсов %d' % (kind, len(out), len(next(iter(out.values()))) if out else 0))
        return out

    def member_triage_acc(d):
        t = true_label(d)
        out = {}
        for p in participants(d):
            lab = d['r1_emergency__' + p].map(norm_label)
            ok = lab.notna() & t.notna()
            out[p] = float((lab[ok] == t[ok]).mean()) if ok.any() else np.nan
        return pd.Series(out)

    def rd(rel, need_score=True):
        p = path_join(ROOT, rel)
        if not os.path.exists(p):
            return None
        d = normalize_newlines(pd.read_csv(p, sep=';', encoding='utf-8-sig', low_memory=False))
        if 'table_source' in d.columns:
            d = d[d['table_source'].fillna('result_df') == 'result_df']
        if need_score and 'Judge_score' not in d.columns:
            return None
        sub = ['Judge_score', 'Vignette'] if need_score else ['Vignette']
        d = d.dropna(subset=sub).drop_duplicates('Vignette', keep='first')
        return d.set_index('Vignette')

    def load_arms(design='noroles', verbose=True):
        a = {}
        for name, rel in arm_paths(design).items():
            d = rd(rel)
            if d is not None:
                a[name] = d
        if verbose:
            print('плеч загружено: %d' % len(a))
            print('   ' + ', '.join(a))
        return a

    def js(arms, name):
        return pd.to_numeric(arms[name]['Judge_score'], errors='coerce').dropna()

    def score_matrix(arms, names=None):
        names = list(arms) if names is None else names
        m = pd.concat([js(arms, n).rename(n) for n in names], axis=1)
        return m.dropna()

    def meta(arms=None):
        d = rd(PBASE_DIR + '/llm_as_judge/delphi_base_noroles_judge.csv')
        cols = [c for c in CASE_COVARS if c in d.columns]
        return d[cols].copy()

    def item_frame(d):
        r = {}
        if d is None or 'judge_details' not in d.columns:
            return r
        for v, s in d['judge_details'].dropna().items():
            try:
                arr = json.loads(s)
            except Exception:
                continue
            for it in arr:
                ax = next((t.split(':', 1)[1] for t in it.get('tags') or [] if isinstance(t, str) and t.startswith('axis:')), None)
                r[v, it['id']] = {'met': bool(it.get('met')), 'points': it.get('points', 0) or 0, 'axis': ax}
        return r

    def load_items(design='noroles', arms_wanted=None, verbose=True):
        paths = arm_paths(design)
        if arms_wanted:
            paths = {k: v for k, v in paths.items() if k in arms_wanted}
        items = {}
        for name, rel in paths.items():
            d = rd(rel)
            f = item_frame(d)
            if f:
                items[name] = f
        common = None
        for f in items.values():
            common = set(f) if common is None else common & set(f)
        ref = next(iter(items.values()))
        pos = sorted([k for k in common or set() if ref[k]['points'] > 0])
        if verbose:
            print('плеч с попунктными вердиктами: %d' % len(items))
            print('общих положительных пунктов: %d' % len(pos))
        return (items, pos)

    def items_long(items, pos, arms_wanted=None):
        names = arms_wanted or list(items)
        rows = []
        for a in names:
            f = items[a]
            for k in pos:
                it = f[k]
                rows.append({'case': k[0], 'cid': k[1], 'arm': a, 'met': int(it['met']), 'points': it['points'], 'axis': it['axis']})
        return pd.DataFrame(rows)

    def paired(a, b):
        ix = a.index.intersection(b.index)
        x, y = (a.loc[ix].astype(float), b.loc[ix].astype(float))
        d = x - y
        p = stats.wilcoxon(x, y)[1] if (d != 0).any() else np.nan
        t = stats.ttest_rel(x, y)
        return {'n': len(ix), 'delta': float(d.mean()), 'sd': float(d.std(ddof=1)), 'share_gt': float((d > 0).mean()), 'p_wilcoxon': float(p), 't': float(t.statistic), 'p_ttest': float(t.pvalue)}

    def se_clt(x):
        x = np.asarray(x, float)
        n = len(x)
        return float(np.sqrt(x.var(ddof=1) / n))

    def se_clustered(x, cluster, cr1=True):
        x = np.asarray(x, float)
        cl = np.asarray(cluster)
        n = len(x)
        e = x - x.mean()
        df = pd.DataFrame({'e': e, 'c': cl})
        r = df.groupby('c', observed=True)['e'].sum().to_numpy()
        ss = df.groupby('c', observed=True)['e'].apply(lambda s: float((s ** 2).sum())).to_numpy()
        cross = float((r ** 2 - ss).sum())
        v = se_clt(x) ** 2 + cross / n ** 2
        G = len(r)
        if cr1 and G > 1:
            v *= G / (G - 1.0)
        return (float(np.sqrt(max(v, 0.0))), G)

    def cluster_test(d, cluster, cr1=True):
        d = pd.Series(d).astype(float)
        cl = pd.Series(cluster).reindex(d.index)
        ok = cl.notna()
        d, cl = (d[ok], cl[ok])
        m = float(d.mean())
        s0 = se_clt(d)
        s1, G = se_clustered(d, cl, cr1=cr1)
        dfree = max(G - 1, 1)
        t = m / s1 if s1 > 0 else np.nan
        p = float(2 * stats.t.sf(abs(t), dfree)) if np.isfinite(t) else np.nan
        tc = stats.t.ppf(0.975, dfree)
        return {'n': len(d), 'G': G, 'mean': m, 'se_naive': s0, 'se_cluster': s1, 'DEFF': (s1 / s0) ** 2 if s0 > 0 else np.nan, 't': float(t), 'df': dfree, 'p': p, 'ci_lo': m - tc * s1, 'ci_hi': m + tc * s1, 'p_naive': float(2 * stats.norm.sf(abs(m / s0))) if s0 > 0 else np.nan}

    def icc_anova(values, groups):
        df = pd.DataFrame({'y': np.asarray(values, float), 'g': np.asarray(groups)})
        df = df.dropna()
        grp = df.groupby('g', observed=True)['y']
        ni = grp.size().to_numpy().astype(float)
        k = len(ni)
        N = ni.sum()
        if k < 2 or N <= k:
            return (0.0, float(ni.mean() if k else 1), k)
        gm = df['y'].mean()
        msb = float((ni * (grp.mean().to_numpy() - gm) ** 2).sum() / (k - 1))
        ssw = float(((df['y'] - df['g'].map(grp.mean()).to_numpy()) ** 2).sum())
        msw = ssw / (N - k)
        B0 = float((N - (ni ** 2).sum() / N) / (k - 1))
        rho = (msb - msw) / (msb + (B0 - 1) * msw) if msb + (B0 - 1) * msw > 0 else 0.0
        return (float(np.clip(rho, 0, 1)), B0, k)

    def ci_wald(p, n, z=1.96):
        se = np.sqrt(p * (1 - p) / n)
        return (p - z * se, p + z * se)

    def ci_wilson(p, n, z=1.96):
        d = 1 + z ** 2 / n
        c = (p + z ** 2 / (2 * n)) / d
        h = z * np.sqrt(p * (1 - p) / n + z ** 2 / (4 * n ** 2)) / d
        return (c - h, c + h)

    def ci_agresti_coull(x, n, z=1.96):
        nt = n + z ** 2
        pt = (x + z ** 2 / 2) / nt
        se = np.sqrt(pt * (1 - pt) / nt)
        return (pt - z * se, pt + z * se)

    def ci_clopper_pearson(x, n, alpha=0.05):
        lo = 0.0 if x == 0 else stats.beta.ppf(alpha / 2, x, n - x + 1)
        hi = 1.0 if x == n else stats.beta.ppf(1 - alpha / 2, x + 1, n - x)
        return (float(lo), float(hi))

    def boot_ci(x, stat=np.mean, B=5000, seed=0, alpha=0.05):
        rng = np.random.default_rng(seed)
        x = np.asarray(x, float)
        n = len(x)
        s = np.array([stat(x[rng.integers(0, n, n)]) for _ in range(B)])
        return (float(np.quantile(s, alpha / 2)), float(np.quantile(s, 1 - alpha / 2)))

    def boot_ci_cluster(x, cluster, stat=np.mean, B=5000, seed=0, alpha=0.05):
        rng = np.random.default_rng(seed)
        df = pd.DataFrame({'x': np.asarray(x, float), 'c': np.asarray(cluster)})
        groups = [g['x'].to_numpy() for _, g in df.groupby('c', observed=True)]
        G = len(groups)
        s = []
        for _ in range(B):
            pick = rng.integers(0, G, G)
            s.append(stat(np.concatenate([groups[i] for i in pick])))
        s = np.array(s)
        return (float(np.quantile(s, alpha / 2)), float(np.quantile(s, 1 - alpha / 2)))

    def holm(pvals):
        p = np.asarray(pvals, float)
        m = len(p)
        order = np.argsort(p)
        adj = np.empty(m)
        run = 0.0
        for i, k in enumerate(order):
            run = max(run, (m - i) * p[k])
            adj[k] = min(run, 1.0)
        return adj

    def bh(pvals):
        p = np.asarray(pvals, float)
        m = len(p)
        order = np.argsort(p)
        adj = np.empty(m)
        run = 1.0
        for i in range(m - 1, -1, -1):
            k = order[i]
            run = min(run, m * p[k] / (i + 1))
            adj[k] = min(run, 1.0)
        return adj

    def bonferroni(pvals):
        p = np.asarray(pvals, float)
        return np.minimum(p * len(p), 1.0)

    def mcnemar(a, b):
        ix = a.index.intersection(b.index)
        x, y = (a.loc[ix].astype(bool), b.loc[ix].astype(bool))
        n01, n10 = (int((~x & y).sum()), int((x & ~y).sum()))
        if n01 + n10 == 0:
            return {'n01': n01, 'n10': n10, 'p': 1.0}
        return {'n01': n01, 'n10': n10, 'p': float(stats.binomtest(n10, n01 + n10, 0.5).pvalue)}
    ORD = {'SELF': 0, 'NONEMERG': 1, 'EMERG': 2}
    GT = {'Self-care appropriate': 'SELF', 'Requires non-emergent care': 'NONEMERG', 'Requires emergent care': 'EMERG'}
    _DASH = dict.fromkeys(map(ord, '‐‑‒–—―−\xad'), '-')
    _KEYS = [('requires emergent care', 'EMERG'), ('requires non-emergent care', 'NONEMERG'), ('self-care appropriate', 'SELF')]

    def norm_label(x):
        if not isinstance(x, str) or not x.strip():
            return None
        if x.strip() in GT:
            return GT[x.strip()]
        import unicodedata
        s = unicodedata.normalize('NFKC', x).translate(_DASH).lower()
        s = re.sub('\\s+', ' ', re.sub('[^a-z\\- ]', ' ', s)).strip()
        for k, v in _KEYS:
            if s.startswith(k) or k in s:
                return v
        if 'non-emergent' in s or 'non emergent' in s:
            return 'NONEMERG'
        if 'emergent' in s or 'emergency' in s:
            return 'EMERG'
        if 'self' in s:
            return 'SELF'
        return None
    _PAT = re.compile('emergency\\s*type\\s*[:\\-]\\s*[\\*_`#\\s]*([^\\n\\r\\*_|]{0,90})', re.I)

    def parse_triage(text):
        if not isinstance(text, str) or not text.strip():
            return None
        import unicodedata
        m = _PAT.search(unicodedata.normalize('NFKC', text).translate(_DASH))
        if not m:
            return None
        return norm_label('Requires ' + m.group(1) if False else m.group(1))

    def pred_triage(d):
        if 'delphi_consensus_emergency_type' in d.columns:
            s = d['delphi_consensus_emergency_type'].map(norm_label)
            if s.notna().mean() > 0.5:
                return s
        for c in ('winner_protocol', 'baseline_protocol_anonymous'):
            if c in d.columns:
                s = d[c].map(parse_triage)
                if s.notna().mean() > 0.3:
                    return s
        return pd.Series(index=d.index, dtype=object)

    def participants(d):
        return sorted((c.split('__', 1)[1] for c in d.columns if c.startswith('r1_emergency__')))

    def true_label(d):
        return d['Type'].map(GT)

    def words(s):
        return len(re.findall('\\S+', str(s))) if isinstance(s, str) else np.nan

    def fmt_p(p):
        if p is None or (isinstance(p, float) and (not np.isfinite(p))):
            return '  n/a '
        return '%.3f' % p if p >= 0.001 else '%.1e' % p

    def show(df, title=None, floatfmt='%.4f'):
        if title:
            print('\n' + title)
            print('-' * max(len(title), 60))
        with pd.option_context('display.width', 200, 'display.max_columns', 60, 'display.float_format', lambda v: floatfmt % v):
            print(df)

    def save(df, name):
        p = path_join(OUT, name)
        df.to_csv(p, sep=';', encoding='utf-8-sig')
        print('сохранено: out/%s  (%d строк)' % (name, len(df)))
        return p
    return locals()
globals().update({k: v for k, v in _build_data_tools().items() if not k.startswith('_')})
del _build_data_tools
RND = sweep_arms('rounds')
print()
OPT = RND['delphi_optimum_early_stop']
NOS = RND['delphi_rounds_no_early_stop']


свип rounds: плеч 2, кейсов 60



In [2]:
a = pd.to_numeric(OPT['Judge_score'], errors='coerce')
b = pd.to_numeric(NOS['Judge_score'], errors='coerce')
ix = a.dropna().index.intersection(b.dropna().index)
d = b.loc[ix] - a.loc[ix]
print('общих кейсов: %d' % len(ix))
print('с ранним остановом : %.4f' % a.loc[ix].mean())
print('без раннего останова: %.4f' % b.loc[ix].mean())
print('разница (без − с)  : %+.4f  (SE %.4f)' % (d.mean(), se_clt(d)))
print('Уилкоксон p = %s' % fmt_p(stats.wilcoxon(d)[1] if (d != 0).any() else np.nan))
n = len(d)
m_, se_ = (float(d.mean()), se_clt(d))
p_tost = max(float(stats.t.sf((m_ + 0.05) / se_, n - 1)), float(stats.t.cdf((m_ - 0.05) / se_, n - 1)))
print('TOST ±0.05: p = %s  ->  %s' % (fmt_p(p_tost), 'эквивалентны' if p_tost < 0.05 else 'эквивалентность не доказана'))
rows = []
for name, dd in [('с ранним остановом', OPT), ('без раннего останова', NOS)]:
    rr = pd.to_numeric(dd.get('delphi_rounds_executed'), errors='coerce')
    tot = [c for c in dd.columns if re.match('r[123]_total_tokens__', c)]
    org = [c for c in ['organizer_after_r1_total_tokens', 'organizer_after_r2_total_tokens', 'final_synthesis_total_tokens'] if c in dd.columns]
    gen = dd[tot].apply(pd.to_numeric, errors='coerce').fillna(0).sum(axis=1)
    agg = dd[org].apply(pd.to_numeric, errors='coerce').fillna(0).sum(axis=1)
    js_ = pd.to_numeric(dd['Judge_score'], errors='coerce')
    t = true_label(dd)
    pr = pred_triage(dd)
    ok = pr.notna() & t.notna()
    rows.append({'режим': name, 'Judge': float(js_.mean()), 'точность триажа': float((pr[ok] == t[ok]).mean()), 'ср. раундов': float(rr.mean()), 'токенов на кейс': float((gen + agg).mean()), 'Judge на 10k': float(js_.mean()) / ((gen + agg).mean() / 10000.0)})
E = pd.DataFrame(rows).set_index('режим')
E.loc['разница'] = E.loc['без раннего останова'] - E.loc['с ранним остановом']
E.loc['рост стоимости, %'] = np.nan
E.loc['рост стоимости, %', 'токенов на кейс'] = 100 * (E.loc['без раннего останова', 'токенов на кейс'] / E.loc['с ранним остановом', 'токенов на кейс'] - 1)
show(E, 'H19.1 Прямое сравнение режимов')


общих кейсов: 60
с ранним остановом : 0.8194
без раннего останова: 0.8258
разница (без − с)  : +0.0064  (SE 0.0165)
Уилкоксон p = 0.894
TOST ±0.05: p = 0.005  ->  эквивалентны

H19.1 Прямое сравнение режимов
------------------------------------------------------------
                      Judge  точность триажа  ср. раундов  токенов на кейс  Judge на 10k
режим                                                                                   
с ранним остановом   0.8194           0.8167       1.7000       26052.2833        0.3145
без раннего останова 0.8258           0.7833       3.0000       44437.5333        0.1858
разница              0.0064          -0.0333       1.3000       18385.2500       -0.1287
рост стоимости, %       NaN              NaN          NaN          70.5706           NaN


In [3]:
STAGES = {}
for pname, (folder, tpl) in ROUNDS_DIR.items():
    fin = rd(arm_paths().get(pname, ''))
    if fin is None:
        continue
    st = {'final': pd.to_numeric(fin['Judge_score'], errors='coerce')}
    for r in (1, 2):
        d = rd(folder + '/llm_as_judge/round_evaluation/' + tpl % r)
        if d is not None:
            st['r%d' % r] = pd.to_numeric(d['Judge_score'], errors='coerce')
    if len(st) == 3:
        rr = pd.to_numeric(fin.get('delphi_rounds_executed'), errors='coerce')
        lab = pd.DataFrame({p: fin['r1_emergency__' + p].map(norm_label) for p in participants(fin)})
        full = lab.dropna(how='all')
        unan1 = (lab.nunique(axis=1) == 1) & lab.notna().all(axis=1)
        STAGES[pname] = pd.DataFrame({'r1': st['r1'], 'r2': st['r2'], 'final': st['final'], 'rounds': rr, 'unan_r1': unan1}).dropna(subset=['r1', 'r2', 'final'])
print('панелей со стадийными оценками:', len(STAGES))
for k, v in STAGES.items():
    print('  %-20s кейсов %d,  остановились после R1: %d' % (k, len(v), int((v['rounds'] == 1).sum())))


панелей со стадийными оценками: 7
  P-base               кейсов 229,  остановились после R1: 120
  P-weak               кейсов 229,  остановились после R1: 119
  P-mixed              кейсов 229,  остановились после R1: 142
  P-strong             кейсов 229,  остановились после R1: 128
  homog gpt            кейсов 229,  остановились после R1: 173
  homog gemini         кейсов 229,  остановились после R1: 192
  homog deepseekv4     кейсов 229,  остановились после R1: 171


In [4]:
rows = []
for pname, S in STAGES.items():
    act = np.where(S['rounds'] == 1, S['r1'], np.where(S['rounds'] == 2, S['r2'], S['final']))
    rules = {'всегда R1': S['r1'].to_numpy(), 'всегда R2': S['r2'].to_numpy(), 'всегда финал': S['final'].to_numpy(), 'фактическое': act, 'единогласие R1': np.where(S['unan_r1'], S['r1'], S['final']), 'оракул стадий': S[['r1', 'r2', 'final']].max(axis=1).to_numpy()}
    for name, v in rules.items():
        rows.append({'панель': pname, 'правило': name, 'Judge': float(np.mean(v)), 'ср. раундов (оценка)': {'всегда R1': 1.0, 'всегда R2': 2.0, 'всегда финал': float(S['rounds'].mean()), 'фактическое': float(S['rounds'].mean()), 'единогласие R1': float(np.where(S['unan_r1'], 1, S['rounds']).mean()), 'оракул стадий': np.nan}[name]})
R = pd.DataFrame(rows)
P = R.pivot(index='панель', columns='правило', values='Judge')
P = P[['всегда R1', 'всегда R2', 'фактическое', 'всегда финал', 'единогласие R1', 'оракул стадий']]
show(P, 'Качество при разных правилах останова')
print()
show(P.sub(P['фактическое'], axis=0).drop(columns=['фактическое']), 'Разница к фактическому правилу')



Качество при разных правилах останова
------------------------------------------------------------
правило           всегда R1  всегда R2  фактическое  всегда финал  единогласие R1  оракул стадий
панель                                                                                          
P-base               0.7837     0.7975       0.7899        0.7899          0.7855         0.8129
P-mixed              0.7930     0.7775       0.7864        0.7864          0.7870         0.8081
P-strong             0.8026     0.7939       0.8061        0.8061          0.8009         0.8292
P-weak               0.7473     0.7439       0.7453        0.7453          0.7477         0.7670
homog deepseekv4     0.7857     0.7825       0.7811        0.7811          0.7836         0.7909
homog gemini         0.7157     0.7177       0.7172        0.7172          0.7152         0.7234
homog gpt            0.7575     0.7598       0.7588        0.7588          0.7583         0.7716


Разница к фактическому пр

In [5]:
rows = []
for pname, S in STAGES.items():
    act = pd.Series(np.where(S['rounds'] == 1, S['r1'], np.where(S['rounds'] == 2, S['r2'], S['final'])), index=S.index)
    for name, v in [('всегда R1', S['r1']), ('всегда R2', S['r2']), ('всегда финал', S['final']), ('единогласие R1', pd.Series(np.where(S['unan_r1'], S['r1'], S['final']), index=S.index))]:
        d = (v - act).dropna()
        nz = d != 0
        p = float(stats.wilcoxon(d)[1]) if nz.any() else np.nan
        n_ = len(d)
        m_ = float(d.mean())
        se_ = se_clt(d) if n_ > 1 else np.nan
        ptost = max(float(stats.t.sf((m_ + 0.02) / se_, n_ - 1)), float(stats.t.cdf((m_ - 0.02) / se_, n_ - 1))) if se_ and se_ > 0 else np.nan
        rows.append({'панель': pname, 'правило': name, 'Δ к фактическому': m_, 'изменилось кейсов': int(nz.sum()), 'p': p, 'p TOST ±0.02': ptost})
T = pd.DataFrame(rows)
show(T.set_index(['панель', 'правило']), 'Правила против фактического')



Правила против фактического
------------------------------------------------------------
                                 Δ к фактическому  изменилось кейсов      p  p TOST ±0.02
панель           правило                                                                 
P-base           всегда R1                -0.0062                 85 0.4002        0.0045
                 всегда R2                 0.0075                 33 0.0273        0.0001
                 всегда финал              0.0000                  0    NaN           NaN
                 единогласие R1           -0.0044                 34 0.1559        0.0000
P-weak           всегда R1                 0.0020                 75 0.8348        0.0005
                 всегда R2                -0.0014                 26 0.6026        0.0000
                 всегда финал              0.0000                  0    NaN           NaN
                 единогласие R1            0.0024                 42 0.5075        0.0000
P-mixed   

In [6]:
rows = []
for pname, S in STAGES.items():
    G = S[S['rounds'] > 1]
    if len(G) < 10:
        continue
    act = np.where(G['rounds'] == 2, G['r2'], G['final'])
    rows.append({'панель': pname, 'спорных кейсов': len(G), 'R1': float(G['r1'].mean()), 'R2': float(G['r2'].mean()), 'финал': float(G['final'].mean()), 'фактическое': float(np.mean(act)), 'оракул': float(G[['r1', 'r2', 'final']].max(axis=1).mean()), 'разрыв до оракула': float(G[['r1', 'r2', 'final']].max(axis=1).mean() - np.mean(act))})
D = pd.DataFrame(rows).set_index('панель')
show(D, 'Только кейсы, где протокол шёл дальше R1')
print()
print('«разрыв до оракула» — верхняя граница того, что можно выиграть')
print('идеальным правилом останова. H19.3 подтверждается, если он мал.')



Только кейсы, где протокол шёл дальше R1
------------------------------------------------------------
                  спорных кейсов     R1     R2  финал  фактическое  оракул  разрыв до оракула
панель                                                                                       
P-base                       109 0.7895 0.8183 0.8026       0.8026  0.8509             0.0483
P-weak                       110 0.7740 0.7668 0.7698       0.7698  0.8149             0.0451
P-mixed                       87 0.7989 0.7580 0.7815       0.7815  0.8388             0.0572
P-strong                     101 0.7990 0.7793 0.8070       0.8070  0.8594             0.0524
homog gpt                     56 0.7506 0.7601 0.7560       0.7560  0.8085             0.0526
homog gemini                  37 0.7070 0.7188 0.7163       0.7163  0.7546             0.0383
homog deepseekv4              58 0.8003 0.7877 0.7822       0.7822  0.8211             0.0389

«разрыв до оракула» — верхняя граница того, что мо

In [7]:
rows = []
for pname, (folder, tpl) in ROUNDS_DIR.items():
    fin = rd(arm_paths().get(pname, ''))
    if fin is None or pname not in STAGES:
        continue
    S = STAGES[pname]
    per_round = {}
    for r in (1, 2, 3):
        cols = [c for c in fin.columns if re.match('r%d_total_tokens__' % r, c)]
        per_round[r] = fin[cols].apply(pd.to_numeric, errors='coerce').fillna(0).sum(axis=1)
    org1 = pd.to_numeric(fin.get('organizer_after_r1_total_tokens'), errors='coerce').fillna(0)
    org2 = pd.to_numeric(fin.get('organizer_after_r2_total_tokens'), errors='coerce').fillna(0)
    syn = pd.to_numeric(fin.get('final_synthesis_total_tokens'), errors='coerce').fillna(0)
    ix = S.index.intersection(fin.index)
    cost_actual = (per_round[1] + per_round[2] + per_round[3] + org1 + org2 + syn).loc[ix]
    cost_r1 = (per_round[1] + syn).loc[ix]
    rows.append({'панель': pname, 'токенов фактически': float(cost_actual.mean()), 'токенов при «всегда R1»': float(cost_r1.mean()), 'экономия, %': 100 * (1 - cost_r1.mean() / cost_actual.mean()), 'Δ качества': float(S['r1'].mean() - np.mean(np.where(S['rounds'] == 1, S['r1'], np.where(S['rounds'] == 2, S['r2'], S['final']))))})
show(pd.DataFrame(rows).set_index('панель'), 'Цена раундов: что даёт остановка сразу после R1')



Цена раундов: что даёт остановка сразу после R1
------------------------------------------------------------
                  токенов фактически  токенов при «всегда R1»  экономия, %  Δ качества
панель                                                                                
P-base                    26746.4541               12604.1616      52.8754     -0.0062
P-weak                    24400.3275               11804.7118      51.6207      0.0020
P-mixed                   22758.0306               10886.0000      52.1663      0.0066
P-strong                  23692.7904               10714.9782      54.7754     -0.0035
homog gpt                 20959.7555               12255.2664      41.5295     -0.0013
homog gemini              12681.4236                7969.8341      37.1535     -0.0015
homog deepseekv4          23911.5284               13157.0611      44.9761      0.0046


In [8]:
crit = pd.DataFrame([{'критерий CREDES': 'порог консенсуса задан a priori', 'статус': 'ДА — единогласие по метке срочности, зашито в реализацию', 'что написать в статье': 'явно указать определение и что оно не менялось'}, {'критерий CREDES': 'число раундов объявлено заранее', 'статус': 'ДА — максимум 3', 'что написать в статье': 'указать максимум и правило досрочного выхода'}, {'критерий CREDES': 'обратная связь между раундами описана', 'статус': 'ДА — сводка организатора', 'что написать в статье': 'описать, что именно видит участник'}, {'критерий CREDES': 'обоснован состав панели', 'статус': 'ЧАСТИЧНО — теперь есть B16 (размер) и C15 (разнообразие)', 'что написать в статье': 'сослаться на аблацию размера и на k_eff'}, {'критерий CREDES': 'отсев участников между раундами описан', 'статус': 'НЕ ПРИМЕНИМО — состав фиксирован', 'что написать в статье': 'сказать явно, что отсева нет'}, {'критерий CREDES': 'анализ незавершённого консенсуса', 'статус': 'ПРОБЕЛ — кейсы без консенсуса после R3 не описаны отдельно', 'что написать в статье': 'добавить долю и их качество'}])
show(crit.set_index('критерий CREDES'), 'Чеклист CREDES применительно к протоколу')
rows = []
for pname, S in STAGES.items():
    fin = rd(arm_paths().get(pname, ''))
    esr = fin.get('delphi_early_stop_reason')
    rr = pd.to_numeric(fin.get('delphi_rounds_executed'), errors='coerce')
    js_ = pd.to_numeric(fin['Judge_score'], errors='coerce')
    noc = rr == 3
    rows.append({'панель': pname, 'доля без консенсуса (3 раунда)': float(noc.mean()), 'Judge при консенсусе': float(js_[~noc].mean()), 'Judge без консенсуса': float(js_[noc].mean()), 'Δ': float(js_[noc].mean() - js_[~noc].mean()), 'p': float(stats.mannwhitneyu(js_[noc].dropna(), js_[~noc].dropna()).pvalue) if noc.sum() > 3 else np.nan})
show(pd.DataFrame(rows).set_index('панель'), 'Пробел CREDES: кейсы, не достигшие консенсуса')



Чеклист CREDES применительно к протоколу
------------------------------------------------------------
                                                                                   статус                           что написать в статье
критерий CREDES                                                                                                                          
порог консенсуса задан a priori         ДА — единогласие по метке срочности, зашито в ...  явно указать определение и что оно не менялось
число раундов объявлено заранее                                           ДА — максимум 3    указать максимум и правило досрочного выхода
обратная связь между раундами описана                            ДА — сводка организатора              описать, что именно видит участник
обоснован состав панели                 ЧАСТИЧНО — теперь есть B16 (размер) и C15 (раз...         сослаться на аблацию размера и на k_eff
отсев участников между раундами описан                   НЕ ПРИМЕНИМО


Пробел CREDES: кейсы, не достигшие консенсуса
------------------------------------------------------------
                  доля без консенсуса (3 раунда)  Judge при консенсусе  Judge без консенсуса       Δ      p
панель                                                                                                     
P-base                                    0.2052                0.7919                0.7824 -0.0095 0.5475
P-weak                                    0.2140                0.7426                0.7554  0.0128 0.5224
P-mixed                                   0.1878                0.7843                0.7956  0.0113 0.5968
P-strong                                  0.1834                0.7977                0.8437  0.0460 0.0476
homog gpt                                 0.0699                0.7604                0.7368 -0.0237 0.2952
homog gemini                              0.0437                0.7184                0.6928 -0.0255 0.5225
homog deepseekv4            

In [9]:
save(E.reset_index(), 'early_stopping_policies_direct_experiment.csv')
save(P.reset_index(), 'early_stopping_policies_rules.csv')
save(T, 'early_stopping_policies_rule_tests.csv')
save(D.reset_index(), 'early_stopping_policies_contested.csv')
save(crit, 'early_stopping_policies_credes.csv')


сохранено: out/early_stopping_policies_direct_experiment.csv  (4 строк)
сохранено: out/early_stopping_policies_rules.csv  (7 строк)
сохранено: out/early_stopping_policies_rule_tests.csv  (28 строк)
сохранено: out/early_stopping_policies_contested.csv  (7 строк)
сохранено: out/early_stopping_policies_credes.csv  (6 строк)


'\\\\?\\D:\\research_llmfmed\\research_llmfmed\\1design_alpha\\dataverse_gold\\github_repository\\analyses\\protocol_design\\results\\early_stopping_policies_credes.csv'